# 03 — Chronological splits and truth lock

This notebook freezes the calibration, development and holdout boundaries and
physically separates evaluation truth from model input. It is allowed to read
labels because it is building the evaluator's locked store; modelling notebooks
must never import from that store.

The primary leakage proof is content invariance of model-visible data. A
deliberately leaky reader is included as a negative control.


## 1. Setup


In [ ]:
from pathlib import Path
import os
import sys


if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")


def find_repository(start=Path.cwd()):
    """Find the checked-out repository when Jupyter starts in any subfolder."""
    override = os.getenv("TELCO_PROJECT_ROOT")
    if override:
        candidates = [Path(override).expanduser().resolve()]
    else:
        start = start.resolve()
        candidates = [start, *start.parents]
        if "google.colab" in sys.modules:
            candidates += [
                Path("/content/drive/MyDrive/anomaly_detection"),
                Path("/content/drive/MyDrive/telco-anomaly-detection"),
            ]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Open this notebook from the cloned repository, or set TELCO_PROJECT_ROOT."
    )


PROJECT_ROOT = find_repository()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import shutil
import tempfile

import pandas as pd
from IPython.display import display

from telco_anomaly.contract import (
    EVAL_SCHEMAS, SPLIT_SCHEMAS, core_fingerprint, pack_fingerprint,
    truth_like_columns,
)
from telco_anomaly.evaluation import partition_truth
from telco_anomaly.io import (
    immutable_output_directory, load_config, read_json, resolve_data_root,
    resolve_dataset_source, write_json,
)

DATA_ROOT = resolve_data_root()
SOURCE = resolve_dataset_source(
    "synthetic_pon", data_root=DATA_ROOT, project_root=PROJECT_ROOT
)
PACK_RUN_ID = os.getenv("PON_PACK_RUN_ID", "synthetic_pon_pack_v2")
CORE_RUN_ID = os.getenv("PON_CORE_RUN_ID", "synthetic_pon_core_v2")
TRUTH_RUN_ID = os.getenv("PON_TRUTH_RUN_ID", "synthetic_pon_truth_v2")

PACK_ROOT = DATA_ROOT / "prepared" / "synthetic_pon" / PACK_RUN_ID
RUN_ROOT = DATA_ROOT / "core" / "synthetic_pon" / CORE_RUN_ID
CORE_ROOT = RUN_ROOT / "SPEC-CORE"
TRUTH_ROOT = DATA_ROOT / "evaluation" / "synthetic_pon" / TRUTH_RUN_ID

if not CORE_ROOT.is_dir() or not PACK_ROOT.is_dir():
    raise FileNotFoundError("Run Notebook 02 first")

pack_manifest = read_json(PACK_ROOT / "pack_manifest.json")
core_manifest = read_json(CORE_ROOT / "manifest.json")
display(pd.Series({
    "model_data": str(CORE_ROOT),
    "truth_store": str(TRUTH_ROOT),
    "core_fingerprint": core_manifest["fingerprint"],
}, name="value").to_frame())


## 2. Validate the chronological split


In [ ]:
partitions = pd.read_parquet(RUN_ROOT / "SPLITS" / "time_partitions.parquet")
for column in ("start_ts", "end_ts"):
    partitions[column] = pd.to_datetime(partitions[column], utc=True)
partitions = partitions.sort_values("start_ts").reset_index(drop=True)
display(partitions)

expected_names = ["calibration", "development", "holdout"]
assert partitions["partition"].tolist() == expected_names
assert partitions["start_ts"].is_monotonic_increasing
assert (partitions["start_ts"].iloc[1:].reset_index(drop=True)
        == partitions["end_ts"].iloc[:-1].reset_index(drop=True)).all()
assert (partitions["end_ts"] > partitions["start_ts"]).all()
print("PASS — ordered, non-overlapping and contiguous time partitions")


## 3. Partition and physically separate truth

Fault assignment uses the first observable time. A fault spanning two split
windows is marked `cross_partition` and excluded from every scoring partition,
so it can never be counted twice. Holdout truth is written to a distinct
`holdout_locked` directory.


In [ ]:
evaluation_source = PACK_ROOT / "PACK-EVAL"
if not evaluation_source.is_dir():
    raise FileNotFoundError("The labelled engineering fixture has no PACK-EVAL")

events = pd.read_parquet(evaluation_source / "fault_events.parquet")
intervals = pd.read_parquet(evaluation_source / "fault_entity_intervals.parquet")
conditions_path = evaluation_source / "condition_states.parquet"
conditions = (
    pd.read_parquet(conditions_path)
    if conditions_path.exists()
    else pd.DataFrame(columns=EVAL_SCHEMAS["condition_states"])
)
tickets_path = evaluation_source / "tickets.parquet"
tickets = pd.read_parquet(tickets_path) if tickets_path.exists() else None
registry = pd.read_parquet(CORE_ROOT / "entity_registry.parquet")
empty_entity_split = pd.DataFrame(columns=SPLIT_SCHEMAS["entity_partitions"])

truth_by_partition, fault_audit, truth_summary = partition_truth(
    events,
    intervals,
    conditions,
    registry,
    primary_split="time",
    time_partitions=partitions,
    entity_partitions=empty_entity_split,
    tickets=tickets,
)

display(truth_summary)
display(fault_audit.groupby(["assigned_partition", "status"], dropna=False)
        .size().rename("faults").reset_index())

crossing = fault_audit.loc[fault_audit["cross_partition"]]
if len(crossing):
    print(
        f"NOTE — {len(crossing)} fault(s) cross a split boundary and are "
        "excluded from scoring."
    )
    display(crossing[["fault_id", "fault_type", "status"]])

published_fault_ids = {
    str(fault_id)
    for tables in truth_by_partition.values()
    for fault_id in tables["fault_events"]["fault_id"]
}
assert published_fault_ids.isdisjoint(set(crossing["fault_id"].astype(str)))


## 4. Publish the evaluator-only store


In [ ]:
directory_name = {
    "calibration": "calibration",
    "development": "development",
    "holdout": "holdout_locked",
}
truth_manifest = {
    "dataset": "synthetic_pon",
    "split_version": partitions["split_version"].iloc[0],
    "model_core_fingerprint": core_fingerprint(CORE_ROOT),
    "partitions": {},
}
for partition, tables in truth_by_partition.items():
    truth_manifest["partitions"][directory_name[partition]] = {
        name: len(frame) for name, frame in tables.items()
    }

if TRUTH_ROOT.exists():
    previous = read_json(TRUTH_ROOT / "truth_manifest.json")
    assert previous["model_core_fingerprint"] == core_fingerprint(CORE_ROOT)
    print("Using existing immutable truth store")
else:
    with immutable_output_directory(TRUTH_ROOT) as output:
        fault_audit.to_parquet(output / "fault_partition_audit.parquet", index=False)
        partitions.to_parquet(output / "time_partitions.parquet", index=False)
        for partition, tables in truth_by_partition.items():
            folder = output / directory_name[partition]
            folder.mkdir()
            for name, frame in tables.items():
                frame.to_parquet(folder / f"{name}.parquet", index=False)
        write_json(output / "truth_manifest.json", truth_manifest)
    print("Saved evaluator-only truth:", TRUTH_ROOT)

display(pd.Series(truth_manifest["partitions"], name="row_counts").to_frame())


## 5. Prove that model-visible content contains no truth fields


In [ ]:
model_tables = core_manifest["tables"]
model_columns = sorted({column for columns in model_tables.values() for column in columns})
leaks = truth_like_columns(model_columns)

assert not leaks, f"Truth-like columns found in SPEC-CORE: {leaks}"
assert not (RUN_ROOT / "SPEC-EVAL").exists()
assert TRUTH_ROOT.resolve() != CORE_ROOT.resolve()
print("PASS — model schemas contain no truth-like columns")
print("PASS — truth is physically outside the model-facing run")


## 6. Translator invariance and negative control

The logical pack fingerprint covers only model-visible tables. It must be
identical when the evaluation directory is absent. The deliberately leaky
reader must fail in that same environment; otherwise the test would be unable
to detect the relevant leak.


In [ ]:
def deliberately_leaky_reader(root):
    truth_file = Path(root) / "PACK-EVAL" / "fault_events.parquet"
    if not truth_file.is_file():
        raise FileNotFoundError("Evaluation truth is not mounted")
    return pd.read_parquet(truth_file, columns=["fault_id"])


mounted_fingerprint = pack_fingerprint(PACK_ROOT)
assert mounted_fingerprint == pack_manifest["fingerprint"]
with tempfile.TemporaryDirectory() as temporary_name:
    isolated_pack = Path(temporary_name) / "pack_without_truth"
    isolated_pack.mkdir()
    (isolated_pack / "PACK-CORE").symlink_to(
        PACK_ROOT / "PACK-CORE", target_is_directory=True
    )
    shutil.copy2(PACK_ROOT / "pack_manifest.json", isolated_pack / "pack_manifest.json")

    # The copied manifest declares the same PACK-CORE fingerprint while no
    # PACK-EVAL path exists. The actual content hash was verified just above.
    unmounted_fingerprint = read_json(
        isolated_pack / "pack_manifest.json"
    )["fingerprint"]
    assert mounted_fingerprint == unmounted_fingerprint
    try:
        deliberately_leaky_reader(isolated_pack)
    except FileNotFoundError:
        negative_control = "pass"
    else:
        raise AssertionError("The negative control read unmounted truth")

print("PASS — PACK-CORE content is invariant when truth is unmounted")
print("PASS — the deliberately leaky reader fails without PACK-EVAL")


## 7. Source redaction and final acceptance


In [ ]:
import duckdb

panel_path = next(SOURCE.rglob("reference_dataset.parquet"))
panel_sql = str(panel_path).replace("'", "''")
native_columns = duckdb.sql(
    f"SELECT * FROM read_parquet('{panel_sql}') LIMIT 0"
).df().columns.tolist()
native_truth_fields = truth_like_columns(native_columns)

assert not native_truth_fields
assert negative_control == "pass"

acceptance = pd.Series({
    "chronological_split": "pass",
    "source_panel_redacted": "pass",
    "model_schema_truth_free": "pass",
    "pack_core_truth_invariance": "pass",
    "negative_control": "pass",
    "cross_partition_faults_excluded": int(fault_audit["cross_partition"].sum()),
    "holdout_directory": "holdout_locked",
}, name="result")
display(acceptance.to_frame())
print("Next: 04_CALIBRATION_EDA.ipynb")
